# 03 Cluster Exploration - Deep Dive Analysis

This notebook demonstrates:
- Deep analysis of individual clusters
- Company-level insights within clusters
- Feature importance and interpretation
- Outlier detection within clusters

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Data

In [ ]:
# Load assignments and profiles
df_assignments = pd.read_csv('../output/germany/01_data/assignments.csv')
df_profiles = pd.read_csv('../output/germany/01_data/profiles.csv')

print(f"✓ Loaded {len(df_assignments)} companies")
print(f"✓ {df_assignments['cluster'].nunique()} clusters")

# Define feature columns
features = ['roa', 'roe', 'ebit_margin', 'gross_margin', 'fcf_margin', 
            'current_ratio', 'debt_to_equity', 'financial_leverage', 'roa_trend']

## 2. Cluster Overview

In [ ]:
# Cluster statistics
cluster_stats = df_assignments.groupby('cluster').agg({
    'gvkey': 'count',
    'cluster_name': 'first'
}).rename(columns={'gvkey': 'n_companies'})

cluster_stats['percentage'] = (cluster_stats['n_companies'] / len(df_assignments) * 100).round(1)

print("📊 Cluster Distribution:\n")
print(cluster_stats)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
cluster_stats['n_companies'].plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Cluster Sizes', fontweight='bold')
ax1.set_xlabel('Cluster')
ax1.set_ylabel('Number of Companies')
ax1.tick_params(axis='x', rotation=0)

# Pie chart
ax2.pie(cluster_stats['n_companies'], labels=cluster_stats['cluster_name'], 
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Cluster Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Select Cluster for Deep Dive

In [ ]:
# Choose cluster to analyze
TARGET_CLUSTER = 2  # Change this to explore different clusters

cluster_companies = df_assignments[df_assignments['cluster'] == TARGET_CLUSTER].copy()
cluster_name = cluster_companies['cluster_name'].iloc[0]

print(f"🎯 Analyzing Cluster {TARGET_CLUSTER}: {cluster_name}")
print(f"   {len(cluster_companies)} companies ({len(cluster_companies)/len(df_assignments)*100:.1f}%)")
print(f"\n📋 Sample Companies:")
print(cluster_companies[['gvkey', 'company_name']].head(10))

## 4. Feature Analysis for Selected Cluster

In [ ]:
# Load detailed data from algorithm results
algo_data_path = Path('../output/germany/02_algorithms/kmeans_comparative/combined/data/combined_data.csv')

if algo_data_path.exists():
    df_detailed = pd.read_csv(algo_data_path)
    cluster_data = df_detailed[df_detailed['cluster'] == TARGET_CLUSTER].copy()
    
    print(f"✓ Loaded detailed data for {len(cluster_data)} companies in cluster")
    
    # Feature statistics
    print(f"\n📊 Feature Statistics (Cluster {TARGET_CLUSTER}):")
    feature_stats = cluster_data[features].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
    print(feature_stats.round(2))
else:
    print("⚠ Detailed data not found - run pipeline first")
    cluster_data = None

In [ ]:
# Feature distribution plots
if cluster_data is not None:
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()
    
    for i, feature in enumerate(features):
        ax = axes[i]
        
        # Histogram with KDE
        cluster_data[feature].hist(bins=20, ax=ax, alpha=0.6, color='steelblue', edgecolor='black')
        ax2 = ax.twinx()
        cluster_data[feature].plot.kde(ax=ax2, color='red', linewidth=2)
        
        ax.set_title(f'{feature.upper()}', fontweight='bold')
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
        ax2.set_ylabel('Density', color='red')
        ax2.tick_params(axis='y', labelcolor='red')
        
        # Add mean line
        mean_val = cluster_data[feature].mean()
        ax.axvline(mean_val, color='green', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
        ax.legend()
    
    plt.suptitle(f'Feature Distributions - Cluster {TARGET_CLUSTER} ({cluster_name})', 
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

## 5. Cluster Comparison

Compare selected cluster against overall population.

In [ ]:
# Compare cluster vs all clusters
if cluster_data is not None:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    key_features = ['roa', 'roe', 'ebit_margin', 'gross_margin', 'fcf_margin', 'debt_to_equity']
    
    for i, feature in enumerate(key_features):
        ax = axes[i]
        
        # Boxplot: Target cluster vs all others
        data_to_plot = [
            df_detailed[df_detailed['cluster'] == TARGET_CLUSTER][feature],
            df_detailed[df_detailed['cluster'] != TARGET_CLUSTER][feature]
        ]
        
        bp = ax.boxplot(data_to_plot, labels=[f'Cluster {TARGET_CLUSTER}', 'Others'],
                        patch_artist=True, showmeans=True)
        bp['boxes'][0].set_facecolor('lightblue')
        bp['boxes'][1].set_facecolor('lightgray')
        
        ax.set_title(feature.upper(), fontweight='bold')
        ax.set_ylabel('Value')
        ax.grid(axis='y', alpha=0.3)
    
    plt.suptitle(f'Cluster {TARGET_CLUSTER} vs Others - Feature Comparison', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Outlier Detection within Cluster

In [ ]:
# Identify outliers using Z-score
if cluster_data is not None:
    z_scores = np.abs(stats.zscore(cluster_data[features], nan_policy='omit'))
    outliers = (z_scores > 3).any(axis=1)
    
    outlier_companies = cluster_data[outliers][['gvkey', 'company_name'] + features]
    
    print(f"🔍 Outliers in Cluster {TARGET_CLUSTER} (Z-score > 3):")
    print(f"   Found {len(outlier_companies)} outlier companies ({len(outlier_companies)/len(cluster_data)*100:.1f}%)")
    
    if len(outlier_companies) > 0:
        print(f"\n📋 Outlier Companies:")
        print(outlier_companies[['gvkey', 'company_name']].head(10))
    else:
        print("   No significant outliers found")

## 7. Feature Correlation within Cluster

In [ ]:
# Correlation heatmap
if cluster_data is not None:
    correlation = cluster_data[features].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title(f'Feature Correlation - Cluster {TARGET_CLUSTER} ({cluster_name})', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Identify strong correlations
    print("\n🔗 Strong Correlations (|r| > 0.7):")
    high_corr = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
    high_corr = high_corr.stack().sort_values(key=abs, ascending=False)
    high_corr = high_corr[abs(high_corr) > 0.7]
    
    if len(high_corr) > 0:
        for (feat1, feat2), corr_val in high_corr.items():
            print(f"   • {feat1} ↔ {feat2}: {corr_val:.3f}")
    else:
        print("   No strong correlations found")

## 8. Company Lookup within Cluster

In [ ]:
# Search for specific company in this cluster
search_term = ""  # Enter company name to search

if search_term:
    matches = cluster_companies[cluster_companies['company_name'].str.contains(search_term, case=False, na=False)]
    
    if len(matches) > 0:
        print(f"✓ Found {len(matches)} match(es) in Cluster {TARGET_CLUSTER}:")
        
        for _, company in matches.iterrows():
            print(f"\n📊 {company['company_name']} (GVKEY: {company['gvkey']})")
            print(f"   Cluster: {company['cluster']} - {company['cluster_name']}")
            
            # Show company features if available
            if cluster_data is not None:
                company_features = cluster_data[cluster_data['gvkey'] == company['gvkey']][features]
                if len(company_features) > 0:
                    print("\n   Key Ratios:")
                    for feat in ['roa', 'roe', 'ebit_margin', 'debt_to_equity']:
                        val = company_features[feat].iloc[0]
                        print(f"     • {feat}: {val:.2f}")
    else:
        print(f"⚠ No matches found for '{search_term}' in this cluster")
else:
    print("💡 Enter a company name in 'search_term' to search within this cluster")

## Next Steps

- **Change `TARGET_CLUSTER`** in cell [3] to explore other clusters
- **04_algorithm_comparison.ipynb**: Compare how different algorithms cluster the same companies
- **99_full_pipeline.ipynb**: Run complete analysis pipeline

## Cluster Interpretation Guide

When interpreting clusters, consider:
1. **Financial Health**: ROA, ROE, margins
2. **Liquidity**: Current ratio
3. **Leverage**: Debt-to-equity, financial leverage
4. **Trends**: ROA trend (growth vs decline)
5. **Outliers**: Companies that don't fit the cluster pattern well